# Grounding DINO - Active Learning Fine-Tune

Bu notebook `active_learning_training.zip` içindeki insan-düzeltilmiş active learning kareleriyle `IDEA-Research/grounding-dino-base` modelini 50 epoch fine-tune eder.

Kareler tam kare olarak değil, `tiled_dino.py` çıkarımındaki döşeme geometrisiyle kesilerek verilir. DINO'nun image processor'ı girdiyi en fazla 1333 uzun kenara indirdiği için 4K bir kare 2.88 kat küçülüyordu; döşemede bu oran 0.87'ye düşüyor ve eğitim seti 73 kareden ~290 döşemeye çıkıyor.

Ayarlar:

- `epochs=50`
- her 10 epochta checkpoint: `checkpoint-epoch-010`, `020`, ...
- `torchrun --nproc_per_node=2`, yani Kaggle T4 x2
- `BATCH_SIZE=1` per GPU ve `GRAD_ACCUM=4`
- `LR=1e-5`, AdamW, grad clip 0.1
- text prompt: `person.`

Süre uyarısı: döşemeyle epoch başına adım sayısı 4 kat arttı, 50 epoch T4 x2'de yaklaşık 3 saat sürer. Kaggle oturum limitini aşarsa `EPOCHS`'u düşür; `checkpoint-epoch-*` klasörleri zaten her 10 epochta yazılıyor.

Not: DINO tarafı YOLO kadar olgun ve risksiz değil. Bu notebook default olarak text backbone ve vision backbone'un bir kısmını dondurur; amaç 93 karelik küçük veriyle modeli tamamen bozma riskini azaltmak ve T4 belleğine sığmaktır.

In [ ]:
import importlib
import os
import subprocess
import sys

packages = ["transformers>=4.40", "accelerate", "pycocotools", "timm", "opencv-python-headless"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU sayisi:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")
else:
    raise SystemExit("GPU yok. Kaggle Settings > Accelerator > GPU T4 x2 secin.")

In [ ]:
%%writefile train_grounding_dino_active.py
import json
import os
import shutil
import time
import zipfile
from collections import defaultdict
from pathlib import Path

import torch
import torch.distributed as dist
from PIL import Image
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor, get_cosine_schedule_with_warmup

ACTIVE_ZIP_NAME = "active_learning_training.zip"
MODEL_NAME = "IDEA-Research/grounding-dino-base"
OUT_DIR = Path("/kaggle/working/runs_dino_active/grounding_dino_active_50ep")
RAW_DIR = Path("/kaggle/working/active_learning_raw_dino")
VAL_VIDEO_PREFIX = "DJI_0574"
TEXT_PROMPT = "person."
EPOCHS = 50
SAVE_PERIOD = 10
BATCH_SIZE = 1          # per GPU. DINO icin guvenli deger; OOM yoksa 2 denenebilir.
GRAD_ACCUM = 4          # effective batch = BATCH_SIZE * gpu_sayisi * GRAD_ACCUM
LR = 1e-5
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
FREEZE_TEXT_BACKBONE = True
FREEZE_VISION_BACKBONE = True

# DINO'nun image processor'i girdiyi en fazla 1333 uzun kenara indiriyor: 4K bir kare 2.88 kat
# kuculuyor. Kareleri cikarimda kullandigimiz doseme geometrisiyle kesince olcek kaybi 0.87'ye
# duser ve egitim seti 73 kareden ~290 dosemeye cikar.
TILE = True
TILE_OVERLAP = 0.2
MIN_BOX_FRAC = 0.4  # kutunun bu kadari doseme icinde kalirsa etiket olarak kullanilir
EMPTY_KEEP = 5      # kutusuz dosemelerin 5'te 1'i negatif ornek olarak tutulur


def setup_dist():
    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:
        dist.init_process_group(backend="nccl")
        rank = int(os.environ["RANK"])
        local_rank = int(os.environ["LOCAL_RANK"])
        world_size = int(os.environ["WORLD_SIZE"])
        torch.cuda.set_device(local_rank)
        return rank, local_rank, world_size
    return 0, 0, 1


def is_main(rank):
    return rank == 0


def find_file(root, name):
    for dirpath, _, filenames in os.walk(root):
        if name in filenames:
            return Path(dirpath) / name
    return None


def prepare_raw(rank):
    zip_path = find_file("/kaggle/input", ACTIVE_ZIP_NAME)
    if zip_path is None:
        raise SystemExit(f"{ACTIVE_ZIP_NAME} bulunamadi. Add Input ile active learning zip'ini ekle.")
    if is_main(rank):
        shutil.rmtree(RAW_DIR, ignore_errors=True)
        RAW_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(RAW_DIR)
        print("Training zip:", zip_path)
    if dist.is_initialized():
        dist.barrier()


def find_image(file_name):
    direct = RAW_DIR / file_name
    if direct.exists():
        return direct
    matches = list(RAW_DIR.rglob(Path(file_name).name))
    if not matches:
        raise FileNotFoundError(file_name)
    return matches[0]


def auto_grid(width, height):
    """tiled_dino.auto_grid ile ayni."""
    long_side = max(width, height)
    if long_side >= 3000:
        return 3, 3
    if long_side >= 1200:
        return 2, 2
    return 1, 1


def tile_windows(width, height, rows, cols, overlap=TILE_OVERLAP):
    """tiled_dino.tile_windows ile ayni."""
    if rows == 1 and cols == 1:
        return [(0, 0, width, height)]
    tile_w = min(width, int(round(width / cols * (1 + overlap))))
    tile_h = min(height, int(round(height / rows * (1 + overlap))))
    step_x = (width - tile_w) / (cols - 1) if cols > 1 else 0
    step_y = (height - tile_h) / (rows - 1) if rows > 1 else 0
    windows = []
    for r in range(rows):
        for c in range(cols):
            x1, y1 = int(round(c * step_x)), int(round(r * step_y))
            windows.append((x1, y1, x1 + tile_w, y1 + tile_h))
    return windows


class CocoPersonDataset(Dataset):
    def __init__(self, split):
        ann_path = RAW_DIR / "annotations" / "instances_default.json"
        coco = json.loads(ann_path.read_text())
        anns_by_image = defaultdict(list)
        for ann in coco["annotations"]:
            if ann.get("iscrowd", 0):
                continue
            anns_by_image[ann["image_id"]].append(ann)

        self.items = []
        empty_seen = 0
        for im in coco["images"]:
            basename = Path(im["file_name"]).name
            video_prefix = basename.rsplit("_f", 1)[0]
            im_split = "val" if video_prefix == VAL_VIDEO_PREFIX else "train"
            if im_split != split:
                continue
            width, height = im["width"], im["height"]
            raw = [a["bbox"] for a in anns_by_image.get(im["id"], []) if a["bbox"][2] > 0 and a["bbox"][3] > 0]
            windows = tile_windows(width, height, *auto_grid(width, height)) if TILE else [(0, 0, width, height)]
            for idx, (x1, y1, x2, y2) in enumerate(windows):
                tile_w, tile_h = x2 - x1, y2 - y1
                boxes = []
                for bx, by, bw, bh in raw:
                    ix1, iy1 = max(bx, x1), max(by, y1)
                    ix2, iy2 = min(bx + bw, x2), min(by + bh, y2)
                    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
                    if inter / (bw * bh) < MIN_BOX_FRAC:
                        continue
                    boxes.append([
                        ((ix1 + ix2) / 2 - x1) / tile_w,
                        ((iy1 + iy2) / 2 - y1) / tile_h,
                        (ix2 - ix1) / tile_w,
                        (iy2 - iy1) / tile_h,
                    ])
                if not boxes:
                    empty_seen += 1
                    # Alt orneklemeyi sadece dosemede yap; tam karede bos kare sayisi cok az.
                    if TILE and empty_seen % EMPTY_KEEP:
                        continue
                self.items.append({
                    "image": find_image(basename),
                    "window": (x1, y1, x2, y2),
                    "boxes": boxes,
                    "name": f"{Path(basename).stem}_t{idx}",
                })

        if split == "val" and not self.items:
            raise SystemExit("Val split bos. DJI_0574 secili veri icinde yoksa VAL_VIDEO_PREFIX'i degistir.")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        image = Image.open(item["image"]).convert("RGB").crop(item["window"])
        boxes = torch.tensor(item["boxes"], dtype=torch.float32).reshape(-1, 4)
        labels = torch.zeros((len(boxes),), dtype=torch.long)
        return image, {"class_labels": labels, "boxes": boxes}, item["name"]


def collate_fn(batch):
    images, labels, names = zip(*batch)
    return list(images), list(labels), list(names)


def freeze_modules(model):
    if not (FREEZE_TEXT_BACKBONE or FREEZE_VISION_BACKBONE):
        return
    for name, param in model.named_parameters():
        lname = name.lower()
        if FREEZE_TEXT_BACKBONE and any(k in lname for k in ("bert", "text", "embeddings")):
            param.requires_grad = False
        if FREEZE_VISION_BACKBONE and any(k in lname for k in ("backbone", "encoder.layers.0", "encoder.layers.1")):
            param.requires_grad = False


def move_labels(labels, device):
    return [{k: v.to(device) for k, v in lab.items()} for lab in labels]


def train_one_epoch(model, processor, loader, optimizer, scheduler, device, epoch, rank):
    model.train()
    total = 0.0
    optimizer.zero_grad(set_to_none=True)
    for step, (images, labels, _) in enumerate(loader, 1):
        inputs = processor(images=images, text=[TEXT_PROMPT] * len(images), return_tensors="pt").to(device)
        labels = move_labels(labels, device)
        outputs = model(**inputs, labels=labels)
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()
        if step % GRAD_ACCUM == 0 or step == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        total += float(loss.detach().cpu()) * GRAD_ACCUM
    return total / max(len(loader), 1)


@torch.no_grad()
def validate(model, processor, loader, device):
    model.eval()
    total = 0.0
    for images, labels, _ in loader:
        inputs = processor(images=images, text=[TEXT_PROMPT] * len(images), return_tensors="pt").to(device)
        labels = move_labels(labels, device)
        outputs = model(**inputs, labels=labels)
        total += float(outputs.loss.detach().cpu())
    return total / max(len(loader), 1)


def save_model(model, processor, path, rank):
    if not is_main(rank):
        return
    path.mkdir(parents=True, exist_ok=True)
    m = model.module if hasattr(model, "module") else model
    m.save_pretrained(path)
    processor.save_pretrained(path)
    print("Kaydedildi:", path, flush=True)


def main():
    rank, local_rank, world_size = setup_dist()
    device = torch.device(f"cuda:{local_rank}" if torch.cuda.is_available() else "cpu")
    prepare_raw(rank)

    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_NAME)
    freeze_modules(model)
    model.to(device)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    if is_main(rank):
        print(f"Parametre: trainable={trainable/1e6:.1f}M / total={total/1e6:.1f}M")

    if world_size > 1:
        model = DDP(model, device_ids=[local_rank], output_device=local_rank, find_unused_parameters=True)

    train_ds = CocoPersonDataset("train")
    val_ds = CocoPersonDataset("val")
    if is_main(rank):
        print(f"Dataset (TILE={TILE}): train={len(train_ds)} val={len(val_ds)}", flush=True)
    train_sampler = DistributedSampler(train_ds, shuffle=True) if world_size > 1 else None
    val_sampler = DistributedSampler(val_ds, shuffle=False) if world_size > 1 else None
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler,
                              shuffle=(train_sampler is None), num_workers=NUM_WORKERS,
                              collate_fn=collate_fn, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, sampler=val_sampler,
                            shuffle=False, num_workers=NUM_WORKERS,
                            collate_fn=collate_fn, pin_memory=True)

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
    steps = max(1, (len(train_loader) + GRAD_ACCUM - 1) // GRAD_ACCUM) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=max(1, steps // 10), num_training_steps=steps)

    history = []
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    for epoch in range(1, EPOCHS + 1):
        if train_sampler is not None:
            train_sampler.set_epoch(epoch)
        start = time.time()
        train_loss = train_one_epoch(model, processor, train_loader, optimizer, scheduler, device, epoch, rank)
        val_loss = validate(model, processor, val_loader, device)
        row = {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "seconds": time.time() - start}
        history.append(row)
        if is_main(rank):
            print(f"epoch {epoch:03d}/{EPOCHS} train={train_loss:.4f} val={val_loss:.4f} time={row['seconds']:.1f}s", flush=True)
            (OUT_DIR / "history.json").write_text(json.dumps(history, indent=2))
        if epoch % SAVE_PERIOD == 0 or epoch == EPOCHS:
            save_model(model, processor, OUT_DIR / f"checkpoint-epoch-{epoch:03d}", rank)

    save_model(model, processor, OUT_DIR / "final", rank)
    if dist.is_initialized():
        dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
# 2 GPU DDP fine-tune. Kaggle'da GPU T4 x2 secili olmali.
!torchrun --standalone --nproc_per_node=2 train_grounding_dino_active.py

In [ ]:
import os
from pathlib import Path

OUT_DIR = Path("/kaggle/working/runs_dino_active/grounding_dino_active_50ep")
print("Checkpointler:")
for p in sorted(OUT_DIR.glob("checkpoint-epoch-*")):
    print(" ", p)
print("Final:", OUT_DIR / "final")
print("History:", OUT_DIR / "history.json")

## Sonraki adım

Fine-tune bitince `final` veya `checkpoint-epoch-050` klasörünü indirip/çıktı olarak bağlayıp, aynı 100 karelik dokunulmamış test GT üzerinde yeniden ölçmeliyiz. Karşılaştırma tablosunda özellikle `AP_small`, `mAP@50`, recall ve inference süresi raporlanacak.